# 🤖 Módulo 17 — Agentes Autônomos e Uso de Ferramentas (Tool Use)

Este módulo mostra como criar **agentes autônomos**, capazes de:

- planejar tarefas
- usar ferramentas externas
- consultar APIs
- acessar bancos de dados
- executar ações
- iterar até atingir um objetivo

Você aprenderá a construir sistemas como:
- AutoGPT simplificado
- Agentes com memória + RAG + ferramentas
- Assistentes que chamam funções automaticamente
- Sistemas que planejam e executam workflows

---

# 🎯 Objetivos do Módulo 17

Ao final deste módulo, você será capaz de:

✔️ entender o conceito de agentes autônomos  
✔️ criar agentes com capacidade de planejamento  
✔️ integrar ferramentas externas (APIs, DB, funções Python)  
✔️ implementar *function calling*  
✔️ criar agentes multi‑passo (loop de raciocínio)  
✔️ construir um mini‑AutoGPT  

---

# 📚 Índice

1. [O que são agentes autônomos?](#conceito)
2. [Arquitetura básica de um agente](#arquitetura)
3. [Function Calling com LLMs](#functions)
4. [Criando ferramentas externas](#tools)
5. [Agente com loop de raciocínio](#loop)
6. [Agente com memória + RAG + ferramentas](#agente-completo)
7. [Projeto Final — Mini AutoGPT](#projeto)

---

# 0. Setup — bibliotecas

Vamos usar:
- Transformers
- Ferramentas Python
- Funções externas simuladas


In [ ]:
import json
import random
from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

<a id="conceito"></a>
# 2. O que são agentes autônomos?

Um **agente autônomo** é um sistema baseado em LLM que:

- recebe um objetivo
- planeja como alcançá-lo
- usa ferramentas externas
- executa ações
- avalia resultados
- repete o ciclo até atingir o objetivo

Diferente de um chatbot comum, um agente:

✔️ age  
✔️ decide  
✔️ executa  
✔️ itera  

Ele não apenas responde — ele **faz**.

---
# 2.1 Diferença entre Chatbot e Agente

| Chatbot | Agente Autônomo |
|--------|------------------|
| Responde perguntas | Resolve objetivos |
| Não usa ferramentas | Usa ferramentas externas |
| Não executa ações | Executa ações reais |
| Não planeja | Planeja passos |
| Não itera | Itera até concluir |

Exemplos:
- ChatGPT padrão → chatbot  
- AutoGPT → agente  
- Devin → agente de programação  
- CrewAI → agentes colaborativos  

---
# 2.2 O ciclo de um agente autônomo

O loop básico é:

```
Objetivo → Planejamento → Ação → Observação → Ajuste → Repetição
```

Esse ciclo é chamado de:

**ReAct (Reason + Act)**  

É o padrão usado por:
- AutoGPT  
- BabyAGI  
- LangChain Agents  
- CrewAI  

---
# 2.3 Exemplo simples de agente (conceitual)

Objetivo:
> "Descobrir a cotação atual do dólar."

Passos do agente:

1. Planejar: "Preciso consultar uma API de câmbio."  
2. Ação: chama a ferramenta `consultar_cotacao()`  
3. Observação: recebe o valor  
4. Resposta final: "O dólar está em R$ 5,12."  

O agente **decide sozinho** qual ferramenta usar.

---
# 2.4 Estrutura básica de um agente

Um agente moderno tem:

- **LLM** → cérebro  
- **Ferramentas** → braços  
- **Memória** → histórico e preferências  
- **RAG** → conhecimento externo  
- **Orquestrador** → controla o loop  

Visualmente:

```
Usuário → Agente → (Ferramentas + Memória + RAG) → Resultado
```

---
# 2.5 Carregando o LLM que será usado no módulo

from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

---
# 2.6 Função simples para o LLM pensar como um agente

Aqui o LLM vai:
- analisar o objetivo
- sugerir um plano de ação

def agente_pensar(objetivo):
    prompt = f"""
Você é um agente autônomo.
Dado o objetivo abaixo, gere um plano de 3 passos para alcançá-lo.

Objetivo: {objetivo}

Plano:
"""
    tokens = tokenizer(prompt, return_tensors="tf")
    output = llm.generate(**tokens, max_length=120)
    return tokenizer.decode(output[0], skip_special_tokens=True)

---
# 2.7 Testando o pensamento do agente

agente_pensar("Aprender o básico de machine learning.")

---
# 2.8 Outro teste

agente_pensar("Organizar minhas finanças pessoais.")

---
# 2.9 O que você aprendeu nesta parte?

✔️ O que é um agente autônomo  
✔️ Diferença entre chatbot e agente  
✔️ O ciclo ReAct (Reason + Act)  
✔️ A arquitetura básica de um agente  
✔️ Como fazer o LLM gerar planos  

Agora estamos prontos para:

**PARTE 3 — Function Calling: ensinando o LLM a usar ferramentas.**

<a id="functions"></a>
# 3. Function Calling — ensinando o LLM a usar ferramentas

O LLM sozinho **não sabe programar ações reais**.

Ele precisa de:
- uma lista de ferramentas disponíveis
- descrições das ferramentas
- um formato para decidir qual ferramenta usar

Vamos implementar isso agora.

---
# 3.1 Definindo ferramentas externas

Vamos criar três ferramentas simples:
- calculadora
- previsão do tempo (simulada)
- consulta de tarefas

def ferramenta_calculadora(expressao):
    try:
        return str(eval(expressao))
    except:
        return "Erro ao calcular."

def ferramenta_clima(cidade):
    return f"O clima em {cidade} está ensolarado com 28°C."

def ferramenta_tarefas():
    return "Suas tarefas: estudar IA, revisar código, caminhar."

---
# 3.2 Catálogo de ferramentas

O LLM precisa saber:
- nome da ferramenta
- descrição
- parâmetros

ferramentas = {
    "calculadora": {
        "descricao": "Resolve expressões matemáticas.",
        "parametros": ["expressao"]
    },
    "clima": {
        "descricao": "Informa o clima de uma cidade.",
        "parametros": ["cidade"]
    },
    "tarefas": {
        "descricao": "Lista tarefas do usuário.",
        "parametros": []
    }
}

---
# 3.3 Carregando o LLM

from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

---
# 3.4 Função para o LLM decidir qual ferramenta usar

O LLM recebe:
- a pergunta
- a lista de ferramentas

E responde:
- "USAR: calculadora | expressao=2+2"
- "USAR: clima | cidade=São Paulo"
- "USAR: tarefas"
- ou "RESPONDER DIRETO"

def decidir_ferramenta(query):
    ferramentas_texto = "\n".join(
        [f"{nome}: {info['descricao']}" for nome, info in ferramentas.items()]
    )

    prompt = f"""
Você é um agente autônomo.
Ferramentas disponíveis:
{ferramentas_texto}

Dado o pedido do usuário, decida se deve usar uma ferramenta.
Formato da resposta:
USAR: <nome> | <parametros>
ou
RESPONDER DIRETO

Pedido: {query}
Decisão:
"""
    tokens = tokenizer(prompt, return_tensors="tf")
    output = llm.generate(**tokens, max_length=80)
    return tokenizer.decode(output[0], skip_special_tokens=True)

---
# 3.5 Testando a decisão do LLM

decidir_ferramenta("Quanto é 12 * 7?")

decidir_ferramenta("Qual é o clima em Salvador?")

decidir_ferramenta("Quais são minhas tarefas hoje?")

decidir_ferramenta("Explique o que é machine learning.")

---
# 3.6 Interpretando a decisão e executando a ferramenta

def executar_ferramenta(decisao):
    if decisao.startswith("USAR:"):
        partes = decisao.replace("USAR:", "").strip().split("|")
        nome = partes[0].strip()

        if nome == "calculadora":
            expressao = partes[1].split("=")[1].strip()
            return ferramenta_calculadora(expressao)

        if nome == "clima":
            cidade = partes[1].split("=")[1].strip()
            return ferramenta_clima(cidade)

        if nome == "tarefas":
            return ferramenta_tarefas()

    return None

---
# 3.7 Pipeline completo de function calling

def agente(query):
    decisao = decidir_ferramenta(query)
    resultado = executar_ferramenta(decisao)

    if resultado:
        return f"[Ferramenta usada]\n{resultado}"

    # Caso não use ferramenta → responder direto
    prompt = f"Responda de forma clara: {query}"
    tokens = tokenizer(prompt, return_tensors="tf")
    output = llm.generate(**tokens, max_length=120)
    return tokenizer.decode(output[0], skip_special_tokens=True)

---
# 3.8 Testando o agente com function calling

agente("Quanto é 15 * 8?")

agente("Qual é o clima em Recife?")

agente("Quais são minhas tarefas?")

agente("Explique o que é deep learning.")

---
# 3.9 O que você aprendeu nesta parte?

✔️ O que é function calling  
✔️ Como ensinar o LLM a escolher ferramentas  
✔️ Como executar ferramentas reais  
✔️ Como integrar tudo em um pipeline  
✔️ Como criar um agente que age, não só responde  

Agora estamos prontos para:

**PARTE 4 — Criando ferramentas externas reais (APIs, DB, Python).**

<a id="tools"></a>
# 4. Criando Ferramentas Externas Reais

Agora vamos criar ferramentas que um agente pode usar de verdade:

✔️ Funções Python  
✔️ Banco de dados  
✔️ APIs simuladas  
✔️ Manipulação de arquivos  

E ensinar o LLM a escolher e usar essas ferramentas.

---
# 4.1 Ferramenta 1 — Leitura de Arquivo

O agente poderá ler arquivos de texto.

import os

def ler_arquivo(caminho):
    if not os.path.exists(caminho):
        return "Arquivo não encontrado."
    with open(caminho, "r", encoding="utf-8") as f:
        return f.read()

Criando um arquivo de exemplo
with open("notas.txt", "w", encoding="utf-8") as f:
    f.write("Minhas notas: estudar IA, revisar código, beber água.")

---
# 4.2 Ferramenta 2 — Consulta SQL real

import sqlite3

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE produtos (
    id INTEGER PRIMARY KEY,
    nome TEXT,
    preco REAL
)
""")

cursor.executemany("""
INSERT INTO produtos (nome, preco)
VALUES (?, ?)
""", [
    ("Notebook", 3500.0),
    ("Mouse", 80.0),
    ("Teclado", 120.0),
])

conn.commit()

def consultar_produto(nome):
    cursor.execute("SELECT * FROM produtos WHERE nome = ?", (nome,))
    row = cursor.fetchone()
    if row:
        return f"Produto: {row[1]}, Preço: R$ {row[2]:.2f}"
    return "Produto não encontrado."

---
# 4.3 Ferramenta 3 — API simulada

Em produção, seria uma chamada HTTP.
Aqui simulamos uma API de câmbio.

def api_cambio(moeda):
    taxas = {
        "USD": 5.12,
        "EUR": 5.45,
        "JPY": 0.034
    }
    return f"1 {moeda} = R$ {taxas.get(moeda, 'desconhecido')}"

---
# 4.4 Catálogo de ferramentas reais

ferramentas = {
    "ler_arquivo": {
        "descricao": "Lê o conteúdo de um arquivo de texto.",
        "parametros": ["caminho"]
    },
    "consultar_produto": {
        "descricao": "Consulta o preço de um produto no banco de dados.",
        "parametros": ["nome"]
    },
    "api_cambio": {
        "descricao": "Retorna a cotação de uma moeda.",
        "parametros": ["moeda"]
    }
}

---
# 4.5 Carregando o LLM

from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

---
# 4.6 LLM decide qual ferramenta usar

def decidir_ferramenta(query):
    ferramentas_texto = "\n".join(
        [f"{nome}: {info['descricao']}" for nome, info in ferramentas.items()]
    )

    prompt = f"""
Você é um agente autônomo.
Ferramentas disponíveis:
{ferramentas_texto}

Dado o pedido do usuário, decida se deve usar uma ferramenta.
Formato da resposta:
USAR: <nome> | <parametros>
ou
RESPONDER DIRETO

Pedido: {query}
Decisão:
"""
    tokens = tokenizer(prompt, return_tensors="tf")
    output = llm.generate(**tokens, max_length=80)
    return tokenizer.decode(output[0], skip_special_tokens=True)

---
# 4.7 Interpretando e executando a ferramenta

def executar_ferramenta(decisao):
    if not decisao.startswith("USAR:"):
        return None

    partes = decisao.replace("USAR:", "").strip().split("|")
    nome = partes[0].strip()

    if nome == "ler_arquivo":
        caminho = partes[1].split("=")[1].strip()
        return ler_arquivo(caminho)

    if nome == "consultar_produto":
        nome_produto = partes[1].split("=")[1].strip()
        return consultar_produto(nome_produto)

    if nome == "api_cambio":
        moeda = partes[1].split("=")[1].strip()
        return api_cambio(moeda)

    return None

---
# 4.8 Pipeline completo do agente com ferramentas reais

def agente(query):
    decisao = decidir_ferramenta(query)
    resultado = executar_ferramenta(decisao)

    if resultado:
        return f"[Ferramenta usada]\n{resultado}"

    # Caso não use ferramenta → responder direto
    prompt = f"Responda de forma clara: {query}"
    tokens = tokenizer(prompt, return_tensors="tf")
    output = llm.generate(**tokens, max_length=120)
    return tokenizer.decode(output[0], skip_special_tokens=True)

---
# 4.9 Testando o agente com ferramentas reais

agente("Leia o arquivo notas.txt")

agente("Qual é o preço do Notebook?")

agente("Qual é a cotação do EUR?")

agente("Explique o que é deep learning.")

---
# 4.10 O que você aprendeu nesta parte?

✔️ Criar ferramentas externas reais  
✔️ Integrar arquivos, SQL e APIs  
✔️ Fazer o LLLM escolher a ferramenta correta  
✔️ Executar a ferramenta e retornar o resultado  
✔️ Construir um agente que interage com o mundo real  

Agora estamos prontos para:

**PARTE 5 — Agente com Loop de Raciocínio (ReAct).**

<a id="loop"></a>
# 5. Agente com Loop de Raciocínio (ReAct)

Agora vamos criar um agente que:

✔️ pensa  
✔️ decide  
✔️ usa ferramentas  
✔️ observa o resultado  
✔️ repete até atingir o objetivo  

Esse é o famoso **ReAct Loop**:

```
Thought → Action → Observation → Thought → Action → ...
```

---
# 5.1 Carregando o LLM

from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

---
# 5.2 Ferramentas disponíveis

def ferramenta_buscar_web(termo):
    return f"Resultado da busca para '{termo}': conteúdo simulado."

def ferramenta_calcular(expr):
    try:
        return str(eval(expr))
    except:
        return "Erro ao calcular."

ferramentas = {
    "buscar_web": {
        "descricao": "Busca informações na web.",
        "parametros": ["termo"]
    },
    "calcular": {
        "descricao": "Resolve expressões matemáticas.",
        "parametros": ["expr"]
    }
}

---
# 5.3 Prompt para o agente pensar e agir

def agente_pensar_e_agir(objetivo, historico):
    ferramentas_texto = "\n".join(
        [f"{nome}: {info['descricao']}" for nome, info in ferramentas.items()]
    )

    prompt = f"""
Você é um agente autônomo usando o padrão ReAct.
Ferramentas disponíveis:
{ferramentas_texto}

Objetivo: {objetivo}

Histórico:
{historico}

Responda no formato:
THOUGHT: <seu raciocínio>
ACTION: <nome_da_ferramenta> | <parametros>
ou
FINAL: <resposta final>

Próxima ação:
"""

    tokens = tokenizer(prompt, return_tensors="tf")
    output = llm.generate(**tokens, max_length=200)
    return tokenizer.decode(output[0], skip_special_tokens=True)

---
# 5.4 Interpretando a ação

def executar_acao(acao):
    if acao.startswith("ACTION:"):
        partes = acao.replace("ACTION:", "").strip().split("|")
        nome = partes[0].strip()

        if nome == "buscar_web":
            termo = partes[1].split("=")[1].strip()
            return ferramenta_buscar_web(termo)

        if nome == "calcular":
            expr = partes[1].split("=")[1].strip()
            return ferramenta_calcular(expr)

    return None

---
# 5.5 Loop ReAct completo

def agente_loop(objetivo, max_passos=5):
    historico = ""

    for passo in range(max_passos):
        resposta = agente_pensar_e_agir(objetivo, historico)

        if "FINAL:" in resposta:
            return resposta.replace("FINAL:", "").strip()

        if "ACTION:" in resposta:
            observacao = executar_acao(resposta)
            historico += f"\n{resposta}\nOBSERVATION: {observacao}\n"
        else:
            historico += f"\n{resposta}\n"

    return "Não consegui completar o objetivo."

---
# 5.6 Testando o agente ReAct

agente_loop("Descubra quanto é (12 * 7) + 5.")

agente_loop("Pesquise sobre inteligência artificial.")

agente_loop("Calcule 45 * 3 e depois busque 'machine learning'.")

---
# 5.7 O que você acabou de construir?

✔️ Um agente que pensa antes de agir  
✔️ Um agente que usa ferramentas automaticamente  
✔️ Um agente que observa resultados  
✔️ Um agente que repete o ciclo até concluir  
✔️ Um agente no estilo AutoGPT / ReAct  

Isso é a base de:
- AutoGPT  
- BabyAGI  
- CrewAI  
- Devin  
- Agentes corporativos modernos  

---
# 5.8 Conclusão desta parte

✔️ Você aprendeu o padrão ReAct  
✔️ Criou um loop de raciocínio completo  
✔️ Implementou pensamento + ação + observação  
✔️ Criou um agente multi‑passo  

Agora estamos prontos para:

**PARTE 6 — Agente Completo: RAG + Memória + Ferramentas + Loop.**

<a id="agente-completo"></a>
# 6. Agente Completo: RAG + Memória + Ferramentas + Loop

Agora vamos integrar:

✔️ RAG (documentos externos)  
✔️ Memória vetorial  
✔️ Ferramentas externas reais  
✔️ Loop ReAct (Reason + Act)  
✔️ LLM como cérebro  

Este é o agente mais completo que você construiu até agora.

---
# 6.1 Carregando o LLM

from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

---
# 6.2 Criando RAG (documentos + FAISS)

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

docs = [
    "Machine learning é uma área da IA que aprende padrões.",
    "Redes neurais são compostas por camadas de neurônios artificiais.",
    "Transformers usam atenção para processar sequências.",
    "Deep learning é um subconjunto do machine learning.",
    "A economia global está passando por transformações tecnológicas."
]

embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_emb = embedder.encode(docs)

dim = doc_emb.shape[1]
rag_index = faiss.IndexFlatL2(dim)
rag_index.add(doc_emb)

def retrieve_docs(query, k=2):
    q_emb = embedder.encode([query])
    distances, indices = rag_index.search(q_emb, k)
    return [docs[i] for i in indices[0]]

---
# 6.3 Criando memória vetorial

memories = [
    "O usuário gosta de gatos.",
    "O usuário trabalha com análise de dados.",
    "O usuário está estudando IA."
]

mem_emb = embedder.encode(memories)
memory_index = faiss.IndexFlatL2(dim)
memory_index.add(mem_emb)

def recall_memory(query, k=2):
    q_emb = embedder.encode([query])
    distances, indices = memory_index.search(q_emb, k)
    return [memories[i] for i in indices[0]]

def add_memory(text):
    memories.append(text)
    new_emb = embedder.encode([text])
    memory_index.add(new_emb)

---
# 6.4 Ferramentas externas reais

import sqlite3
import os

Banco de dados
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE produtos (
    id INTEGER PRIMARY KEY,
    nome TEXT,
    preco REAL
)
""")

cursor.executemany("""
INSERT INTO produtos (nome, preco)
VALUES (?, ?)
""", [
    ("Notebook", 3500.0),
    ("Mouse", 80.0),
    ("Teclado", 120.0),
])

conn.commit()

def consultar_produto(nome):
    cursor.execute("SELECT * FROM produtos WHERE nome = ?", (nome,))
    row = cursor.fetchone()
    if row:
        return f"Produto: {row[1]}, Preço: R$ {row[2]:.2f}"
    return "Produto não encontrado."

Arquivo
with open("notas.txt", "w", encoding="utf-8") as f:
    f.write("Minhas notas: estudar IA, revisar código, beber água.")

def ler_arquivo(caminho):
    if not os.path.exists(caminho):
        return "Arquivo não encontrado."
    return open(caminho, "r", encoding="utf-8").read()

API simulada
def api_cambio(moeda):
    taxas = {"USD": 5.12, "EUR": 5.45, "JPY": 0.034}
    return f"1 {moeda} = R$ {taxas.get(moeda, 'desconhecido')}"

Catálogo
ferramentas = {
    "consultar_produto": {"descricao": "Consulta preço de produto.", "parametros": ["nome"]},
    "ler_arquivo": {"descricao": "Lê arquivo de texto.", "parametros": ["caminho"]},
    "api_cambio": {"descricao": "Consulta cotação de moeda.", "parametros": ["moeda"]}
}

---
# 6.5 Prompt ReAct para o agente completo

def agente_react(objetivo, historico):
    ferramentas_texto = "\n".join(
        [f"{nome}: {info['descricao']}" for nome, info in ferramentas.items()]
    )

    prompt = f"""
Você é um agente autônomo completo.
Ferramentas disponíveis:
{ferramentas_texto}

Objetivo: {objetivo}

Histórico:
{historico}

Use o formato:
THOUGHT: <raciocínio>
ACTION: <ferramenta> | <parametros>
OBSERVATION: <resultado>
ou
FINAL: <resposta final>

Próxima ação:
"""
    tokens = tokenizer(prompt, return_tensors="tf")
    output = llm.generate(**tokens, max_length=250)
    return tokenizer.decode(output[0], skip_special_tokens=True)

---
# 6.6 Interpretando e executando ações

def executar_acao(acao):
    if not acao.startswith("ACTION:"):
        return None

    partes = acao.replace("ACTION:", "").strip().split("|")
    nome = partes[0].strip()

    if nome == "consultar_produto":
        nome_produto = partes[1].split("=")[1].strip()
        return consultar_produto(nome_produto)

    if nome == "ler_arquivo":
        caminho = partes[1].split("=")[1].strip()
        return ler_arquivo(caminho)

    if nome == "api_cambio":
        moeda = partes[1].split("=")[1].strip()
        return api_cambio(moeda)

    return None

---
# 6.7 Loop completo do agente

def agente_completo(objetivo, max_passos=5):
    historico = ""

    for _ in range(max_passos):
        resposta = agente_react(objetivo, historico)

        if "FINAL:" in resposta:
            return resposta.replace("FINAL:", "").strip()

        if "ACTION:" in resposta:
            observacao = executar_acao(resposta)
            historico += f"\n{resposta}\nOBSERVATION: {observacao}\n"
        else:
            historico += f"\n{resposta}\n"

    return "Não consegui completar o objetivo."

---
# 6.8 Testes do agente completo

agente_completo("Descubra o preço do Notebook.")

agente_completo("Leia o arquivo notas.txt e me diga o que contém.")

agente_completo("Qual é a cotação do EUR?")

agente_completo("Explique machine learning usando documentos e memórias.")

---
# 6.9 O que você acabou de construir?

✔️ Um agente com RAG  
✔️ Um agente com memória vetorial  
✔️ Um agente com ferramentas reais  
✔️ Um agente com loop ReAct  
✔️ Um agente capaz de planejar e agir  

Este é praticamente um **AutoGPT simplificado**.

---
# 6.10 Conclusão desta parte

✔️ Você integrou todos os componentes de IA moderna  
✔️ Criou um agente completo e funcional  
✔️ Está pronto para o GRANDE FINAL:

**PARTE 7 — Projeto Final: Mini‑AutoGPT.**

# 7. Projeto Final — Mini‑AutoGPT

Agora você vai construir um agente autônomo completo, com:

✔️ RAG  
✔️ Memória vetorial  
✔️ Ferramentas externas reais  
✔️ Loop ReAct  
✔️ Planejamento multi‑passo  

Este é o seu **Mini‑AutoGPT**.

---
# 7.1 Carregando o LLM

from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

---
# 7.2 RAG — Documentos + FAISS

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

docs = [
    "Machine learning é uma área da IA que aprende padrões.",
    "Redes neurais são compostas por camadas de neurônios artificiais.",
    "Transformers usam atenção para processar sequências.",
    "Deep learning é um subconjunto do machine learning.",
    "A economia global está passando por transformações tecnológicas."
]

embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_emb = embedder.encode(docs)

dim = doc_emb.shape[1]
rag_index = faiss.IndexFlatL2(dim)
rag_index.add(doc_emb)

def retrieve_docs(query, k=2):
    q_emb = embedder.encode([query])
    distances, indices = rag_index.search(q_emb, k)
    return [docs[i] for i in indices[0]]

---
# 7.3 Memória vetorial

memories = [
    "O usuário gosta de gatos.",
    "O usuário trabalha com análise de dados.",
    "O usuário está estudando IA."
]

mem_emb = embedder.encode(memories)
memory_index = faiss.IndexFlatL2(dim)
memory_index.add(mem_emb)

def recall_memory(query, k=2):
    q_emb = embedder.encode([query])
    distances, indices = memory_index.search(q_emb, k)
    return [memories[i] for i in indices[0]]

def add_memory(text):
    memories.append(text)
    new_emb = embedder.encode([text])
    memory_index.add(new_emb)

---
# 7.4 Ferramentas externas reais

import sqlite3
import os

Banco de dados
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE produtos (
    id INTEGER PRIMARY KEY,
    nome TEXT,
    preco REAL
)
""")

cursor.executemany("""
INSERT INTO produtos (nome, preco)
VALUES (?, ?)
""", [
    ("Notebook", 3500.0),
    ("Mouse", 80.0),
    ("Teclado", 120.0),
])

conn.commit()

def consultar_produto(nome):
    cursor.execute("SELECT * FROM produtos WHERE nome = ?", (nome,))
    row = cursor.fetchone()
    if row:
        return f"Produto: {row[1]}, Preço: R$ {row[2]:.2f}"
    return "Produto não encontrado."

Arquivo
with open("notas.txt", "w", encoding="utf-8") as f:
    f.write("Minhas notas: estudar IA, revisar código, beber água.")

def ler_arquivo(caminho):
    if not os.path.exists(caminho):
        return "Arquivo não encontrado."
    return open(caminho, "r", encoding="utf-8").read()

API simulada
def api_cambio(moeda):
    taxas = {"USD": 5.12, "EUR": 5.45, "JPY": 0.034}
    return f"1 {moeda} = R$ {taxas.get(moeda, 'desconhecido')}"

Catálogo
ferramentas = {
    "consultar_produto": {"descricao": "Consulta preço de produto.", "parametros": ["nome"]},
    "ler_arquivo": {"descricao": "Lê arquivo de texto.", "parametros": ["caminho"]},
    "api_cambio": {"descricao": "Consulta cotação de moeda.", "parametros": ["moeda"]}
}

---
# 7.5 Prompt ReAct para o Mini‑AutoGPT

def mini_autogpt_step(objetivo, historico):
    ferramentas_texto = "\n".join(
        [f"{nome}: {info['descricao']}" for nome, info in ferramentas.items()]
    )

    prompt = f"""
Você é um agente autônomo avançado (Mini‑AutoGPT).
Ferramentas disponíveis:
{ferramentas_texto}

Objetivo: {objetivo}

Histórico:
{historico}

Use o formato:
THOUGHT: <raciocínio>
ACTION: <ferramenta> | <parametros>
OBSERVATION: <resultado>
ou
FINAL: <resposta final>

Próxima ação:
"""
    tokens = tokenizer(prompt, return_tensors="tf")
    output = llm.generate(**tokens, max_length=300)
    return tokenizer.decode(output[0], skip_special_tokens=True)

---
# 7.6 Interpretando e executando ações

def executar_acao(acao):
    if not acao.startswith("ACTION:"):
        return None

    partes = acao.replace("ACTION:", "").strip().split("|")
    nome = partes[0].strip()

    if nome == "consultar_produto":
        nome_produto = partes[1].split("=")[1].strip()
        return consultar_produto(nome_produto)

    if nome == "ler_arquivo":
        caminho = partes[1].split("=")[1].strip()
        return ler_arquivo(caminho)

    if nome == "api_cambio":
        moeda = partes[1].split("=")[1].strip()
        return api_cambio(moeda)

    return None

---
# 7.7 Loop completo do Mini‑AutoGPT

def mini_autogpt(objetivo, max_passos=6):
    historico = ""

    for _ in range(max_passos):
        resposta = mini_autogpt_step(objetivo, historico)

        if "FINAL:" in resposta:
            return resposta.replace("FINAL:", "").strip()

        if "ACTION:" in resposta:
            observacao = executar_acao(resposta)
            historico += f"\n{resposta}\nOBSERVATION: {observacao}\n"
        else:
            historico += f"\n{resposta}\n"

    return "Não consegui completar o objetivo."

---
# 7.8 Testes do Mini‑AutoGPT

mini_autogpt("Descubra o preço do Notebook e explique se é caro.")

mini_autogpt("Leia o arquivo notas.txt e gere um resumo.")

mini_autogpt("Qual é a cotação do EUR e como isso afeta a economia?")

mini_autogpt("Explique machine learning usando documentos e memórias.")

---
# 7.9 O que você acabou de construir?

✔️ Um agente autônomo completo  
✔️ Com RAG  
✔️ Com memória vetorial  
✔️ Com ferramentas reais  
✔️ Com loop ReAct  
✔️ Com planejamento multi‑passo  

Isso é, na prática, um **Mini‑AutoGPT**.

---
# 7.10 Conclusão do Módulo 17

✔️ Você domina agentes autônomos  
✔️ Você domina function calling  
✔️ Você domina ferramentas externas  
✔️ Você domina loops ReAct  
✔️ Você construiu um Mini‑AutoGPT  

Parabéns — você concluiu o **Módulo 17**.

O próximo passo é o **Módulo 18 — Fine‑Tuning Avançado + RLHF + Alignment**.